# Bài thực hành làm sạch và phân tích dữ liệu đặt vé máy bay

**Phiên bản Airline 1.0 · Dự án Giữa kỳ**

| Thông tin nhóm | Điền vào đây |
|---|---|
| Mã nhóm | NhomXX |
| Thành viên 1 (MSSV) — Kỹ sư Làm sạch Dữ liệu | Trần Tùng — Giai đoạn A + B1–B5 |
| Thành viên 2 — QA & Data Architect | Tạ Minh Quân — B6–C4 + Câu 1 |
| Thành viên 3 (MSSV) — BI & Data Analyst | … — Câu 2–10 + 8 biểu đồ |
| Ngày chạy lại toàn bộ lần cuối | 2026-10-20 |

**Trước khi nộp:** đảm bảo file tên là `airline_analysis.ipynb`.


## 0 · Thiết lập

Ô 0.1 chứa công cụ hỗ trợ dùng chung. **Không cần sửa**, chỉ cần chạy.


In [1]:
# 0.1 CÔNG CỤ HỖ TRỢ — KHÔNG CẦN SỬA
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 50)

# ---- Đường dẫn: notebook chạy từ notebooks/ nên data nằm ở ../data/ ----
# Theo chuẩn đề bài: data/airline_bookings_dirty.csv (file gốc không được sửa)
CANDIDATES = [
    Path('../data/airline_bookings_dirty.csv'),   # Máy cá nhân (chuẩn)
    Path('data/airline_bookings_dirty.csv'),       # Chạy từ thư mục gốc
    Path('airline_bookings_dirty.csv')             # Colab
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), CANDIDATES[0])
OUTPUT_DIR = Path('outputs') if Path('outputs').exists() else Path('../outputs')
for sub in ['', 'figures', 'tables']:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

# ---- Quy ước của đề (Mục 5) ----
SNAPSHOT_DATE = pd.Timestamp('2026-09-15')
MISSING_TOKENS = ['NULL', 'N/A', 'NA', '?', 'unknown', '-']
AIRLINE_COLORS = {'Aurora Air': '#2F4858', 'Horizon Wings': '#E07A1F'}
STATUS_COLORS = {'Flown': '#5B8E7D', 'Cancelled': '#B81D24', 'No-Show': '#F2A541'}
FIGSIZE = (9, 5)

AIRLINES = ['Aurora Air', 'Horizon Wings']
MEALS = ['Standard', 'Vegetarian', 'Special', 'No Meal']
SEGMENTS = ['Direct', 'Online OTA', 'Offline Agent', 'Corporate', 'Group', 'Package Tour', 'Loyalty']
CHANNELS = ['Website', 'Mobile App', 'Travel Agency', 'GDS/Corporate']
FARE_POLICIES = ['Flexible', 'Semi-Flexible', 'Non-Refundable']
TRAVELERS = ['Individual', 'Linked Booking', 'Corporate Contract', 'Group']
STATUSES = ['Flown', 'Cancelled', 'No-Show']
CABIN_CODES = ['E1', 'E2', 'E3', 'E4', 'P1', 'P2', 'B1', 'B2', 'B3', 'F1', 'F2', 'F3']
LEAD_GROUPS = ['0-7', '8-30', '31-90', '91-180', '181+']

DEPARTURE_MIN, DEPARTURE_MAX = pd.Timestamp('2024-07-01'), pd.Timestamp('2026-08-31')
STATUS_MIN, STATUS_MAX = pd.Timestamp('2024-01-01'), pd.Timestamp('2026-09-15')
LEAD_MIN, LEAD_MAX = 0, 365
WEEKEND_MAX, WEEK_MAX = 20, 50
PASSENGER_MAX = 10
FARE_MIN, FARE_MAX = 0, 5000
REQ_MAX = 5
N_RAW_ROWS = 10325

CLEAN_REQUIRED = ['reservation_id', 'airline', 'is_cancelled', 'booking_lead_days', 'departure_date', 'trip_weekend_days',
                  'trip_weekdays', 'adult_passengers', 'child_passengers', 'infant_passengers', 'meal_option',
                  'passenger_country', 'market_segment', 'sales_channel', 'is_returning_customer', 'booked_cabin_code',
                  'assigned_cabin_code', 'fare_policy', 'traveler_type', 'avg_ticket_fare', 'special_service_requests',
                  'booking_status', 'booking_status_date', 'departure_year', 'departure_month', 'total_trip_days',
                  'total_passengers', 'lead_time_group', 'cabin_changed_flag', 'booking_value_est']
SUMMARY_COLUMNS = ['airline', 'departure_year', 'departure_month', 'n_reservations', 'n_cancelled',
                   'cancel_rate_pct', 'passengers_flown']
LOG_COLUMNS = ['step', 'field', 'issue', 'detection_rule', 'action', 'affected_rows',
               'changed_cells', 'removed_rows', 'example_record_ids', 'reason', 'check_after']
LINEAGE_COLUMNS = ['record_id', 'reservation_id', 'status']
LINEAGE_STATUS = ['kept', 'merged', 'dropped']
QUALITY_COLUMNS = ['column', 'stage', 'denominator_rows', 'missing_count', 'invalid_count', 'missing_pct']

CORE_FILES = ['reservations_clean.csv', 'monthly_summary.csv', 'cleaning_log.csv', 'row_lineage.csv', 'quality_report.csv']
FIGURE_FILES = ['fig01_airline_cancel.png', 'fig02_monthly_departures_2025.png', 'fig03_leadtime_cancel.png',
                'fig04_top10_country.png', 'fig05_segment_cancel.png', 'fig06_fare_policy_status.png',
                'fig07_fare_hist.png', 'fig08_trip_days.png']
TABLE_FILES = ['a_missing_survey.csv', 'a_issue_catalog.csv', 'q01_before_after.csv',
               'q02_airline_cancel.csv', 'q03_monthly_departures_2025.csv', 'q04_leadtime_stats.csv',
               'q04_leadtime_cancel.csv', 'q05_top10_country.csv', 'q06_segment_cancel.csv',
               'q07_fare_policy_status.csv', 'q08_fare_stats_by_airline.csv', 'q09_trip_days.csv',
               'q10_cabin_change_by_airline.csv', 'q10_cabin_change_by_booked_code.csv']


def check(label, fn, hint=''):
    try:
        ok = bool(fn())
        detail = ''
    except Exception as e:
        ok = False
        detail = f'{type(e).__name__}: {e}'
    if ok:
        print('✅ ' + label)
    else:
        extra = (' — ' + detail) if detail else ((' — ' + hint) if hint else '')
        print('❌ ' + label + extra)
    return ok


def clean_key(x):
    if pd.isna(x): return pd.NA
    return re.sub(r'[^a-z]', '', str(x).strip().lower())


def log_step(step, field, issue, detection_rule, action, affected_rows=0, changed_cells=0,
             removed_rows=0, example_record_ids='', reason='', check_after=''):
    cleaning_log_rows.append({
        'step': step, 'field': field, 'issue': issue, 'detection_rule': detection_rule,
        'action': action, 'affected_rows': int(affected_rows), 'changed_cells': int(changed_cells),
        'removed_rows': int(removed_rows), 'example_record_ids': str(example_record_ids),
        'reason': reason, 'check_after': check_after,
    })


def save_table(frame, name):
    if frame is None:
        print(f'⏳ Chưa lưu {name}.csv: biến còn None.')
        return
    frame.to_csv(OUTPUT_DIR / 'tables' / f'{name}.csv', index=False, encoding='utf-8-sig')
    print(f'💾 outputs/tables/{name}.csv ({len(frame)} dòng)')


def save_fig(fig, name):
    if name + '.png' not in FIGURE_FILES:
        print(f'⚠️ Tên {name}.png không có trong danh sách chuẩn.')
    fig.savefig(OUTPUT_DIR / 'figures' / f'{name}.png', dpi=150, bbox_inches='tight')
    print(f'🖼️ outputs/figures/{name}.png')


def finish_chart(ax, title, xlabel, ylabel, note=None):
    ax.set_title(title, fontsize=13, fontweight='bold', loc='left')
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.spines[['top', 'right']].set_visible(False)
    if note:
        ax.figure.text(0.01, -0.04, note, fontsize=9, color='#555555', ha='left', va='top', wrap=True)


def show_progress():
    items = [('Dữ liệu', f, OUTPUT_DIR / f) for f in CORE_FILES] + \
            [('Bảng', f, OUTPUT_DIR / 'tables' / f) for f in TABLE_FILES] + \
            [('Hình', f, OUTPUT_DIR / 'figures' / f) for f in FIGURE_FILES]
    t = pd.DataFrame([(g, f, '✅' if q.exists() else '⬜') for g, f, q in items], columns=['Nhóm', 'Tệp', 'Có'])
    done = (t['Có'] == '✅').sum()
    print(f'Tiến độ tệp đầu ra: {done}/{len(t)} ({done / len(t):.0%})')
    return t


cleaning_log_rows = []
print('✅ Thiết lập xong. DATA_PATH =', DATA_PATH)

✅ Thiết lập xong. DATA_PATH = data/airline_bookings_dirty.csv


In [2]:
# 0.2 ĐỌC DỮ LIỆU — đọc mọi cột ở dạng chuỗi để thấy giá trị gốc
if not DATA_PATH.exists():
    raise FileNotFoundError('Không thấy airline_bookings_dirty.csv.')

raw_df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
print('Kích thước đầu vào:', raw_df.shape)
assert raw_df.shape == (10325, 24), f'Đầu vào phải là 10.325 dòng × 24 cột, hiện tại: {raw_df.shape}'
assert raw_df['record_id'].is_unique, 'record_id phải duy nhất ở dữ liệu đầu vào.'
print('✅ Đầu vào đúng 10.325 dòng × 24 cột')
raw_df.head()

Kích thước đầu vào: (10325, 24)
✅ Đầu vào đúng 10.325 dòng × 24 cột


---
# GIAI ĐOẠN A · KHẢO SÁT TRƯỚC KHI SỬA
*Mục tiêu: hiểu dữ liệu đang sai ở đâu, sai bao nhiêu — **chưa sửa gì cả**.*

**Thành viên 1 phụ trách toàn bộ Giai đoạn A.**

| | |
|---|---|
| **Mục tiêu** | Lập bảng thiếu theo cột và danh mục **ít nhất 10 nhóm vấn đề**, mỗi nhóm có số dòng và tối đa 5 `record_id` ví dụ. |
| **Đầu vào → Đầu ra** | `raw_df` → `missing_survey`, `issue_catalog` |
| **Thành viên phụ trách** | Thành viên 1 |


In [3]:
# A1 · Tổng quan dữ liệu
print('=== HÌNH DẠNG DỮ LIỆU ===')
print('Shape:', raw_df.shape)
print()
print('=== TÊN CỘT ===')
print(list(raw_df.columns))
print()
print('=== KIỂU DỮ LIỆU (đọc dạng str nên đều là object) ===')
print(raw_df.dtypes)
print()
print('=== 5 DÒNG ĐẦU ===')
raw_df.head()

=== HÌNH DẠNG DỮ LIỆU ===
Shape: (10325, 24)

=== TÊN CỘT ===
['record_id', 'reservation_id', 'airline', 'is_cancelled', 'booking_lead_days', 'departure_date', 'trip_weekend_days', 'trip_weekdays', 'adult_passengers', 'child_passengers', 'infant_passengers', 'meal_option', 'passenger_country', 'market_segment', 'sales_channel', 'is_returning_customer', 'booked_cabin_code', 'assigned_cabin_code', 'fare_policy', 'traveler_type', 'avg_ticket_fare', 'special_service_requests', 'booking_status', 'booking_status_date']

=== KIỂU DỮ LIỆU (đọc dạng str nên đều là object) ===
record_id                   object
reservation_id              object
airline                     object
is_cancelled                object
booking_lead_days           object
departure_date              object
trip_weekend_days           object
trip_weekdays               object
adult_passengers            object
child_passengers            object
infant_passengers           object
meal_option                 object
passen

In [4]:
# A2 · Bảng thiếu theo cột
# Đếm ô rỗng (sau strip) và ô là ký hiệu thiếu theo từng cột

records = []
for col in raw_df.columns:
    n_rows = len(raw_df)
    empty_count = (raw_df[col].str.strip() == '').sum()
    token_count = raw_df[col].str.lower().isin([t.lower() for t in MISSING_TOKENS]).sum()
    missing_total = empty_count + token_count
    missing_pct = round(missing_total / n_rows * 100, 2)
    records.append({
        'column': col,
        'n_rows': n_rows,
        'empty_count': int(empty_count),
        'token_count': int(token_count),
        'missing_total': int(missing_total),
        'missing_pct': missing_pct
    })

missing_survey = pd.DataFrame(records)
print(f'missing_survey: {missing_survey.shape}')
missing_survey

missing_survey: (24, 6)


In [5]:
# A3 · Nhãn của các cột phân loại — value_counts
cat_cols = ['airline', 'is_cancelled', 'meal_option', 'market_segment', 'sales_channel',
            'fare_policy', 'traveler_type', 'booking_status', 'booked_cabin_code', 'assigned_cabin_code']

for col in cat_cols:
    print(f'\n{"="*55}')
    print(f'  {col.upper()}')
    print(f'{"="*55}')
    print(raw_df[col].value_counts(dropna=False).to_string())


  AIRLINE
airline
Aurora Air         6637
Horizon Wings      3513
Aurora               35
 Aurora Air          24
aurora air           23
horizon wings        23
AURORA AIR           21
Horizon              19
HORIZON WINGS        18
 Horizon Wings       12

  IS_CANCELLED
is_cancelled
0        7385
1        2817
no         28
False      27
0.0        21
 No        20
 Yes       10
1.0         8
yes         5
True        4

  MEAL_OPTION
meal_option
Standard         6084
No Meal          1879
Vegetarian       1256
Special           944
Std                25
standard meal      19
 Standard          18
STANDARD           15
NO MEAL            13
Veg                 8
 No Meal            7
none                7
?                   6
 Vegetarian         5
VEG                 5
NULL                4
N/A                 4
 Special            4
vegetarian          3
-                   3
                    3
null                3
special meal        2
no_meal             2
SPECIAL          

In [6]:
# A4 · Định dạng ngày, số, tiền và mã quốc gia
print('=== departure_date: các định dạng khác YYYY-MM-DD ===')
dep_nonstandard = raw_df['departure_date'][
    ~raw_df['departure_date'].str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}')
]
print(dep_nonstandard.value_counts().head(25))

print('\n=== avg_ticket_fare: giá trị không phải số thuần ===')
fare_nonnum = raw_df['avg_ticket_fare'][
    pd.to_numeric(raw_df['avg_ticket_fare'], errors='coerce').isna()
]
print(fare_nonnum.value_counts().head(20))

print('\n=== passenger_country: mã không chuẩn ===')
country_bad = raw_df['passenger_country'][
    ~raw_df['passenger_country'].str.strip().str.fullmatch(r'[A-Z]{3}')
]
print(country_bad.value_counts().head(20))

print('\n=== booking_status_date: mẫu ===')
print(raw_df['booking_status_date'].value_counts(dropna=False).head(10))

print('\n=== Số trường có ký hiệu thiếu hoặc rỗng ===')
for col in ['trip_weekend_days', 'trip_weekdays', 'adult_passengers', 'child_passengers',
            'infant_passengers', 'special_service_requests', 'booking_lead_days']:
    unusual = raw_df[col][pd.to_numeric(raw_df[col], errors='coerce').isna()]
    if len(unusual) > 0:
        print(f'{col}: {len(unusual)} giá trị không phải số')
        print(' ', unusual.value_counts().head(5).to_dict())

=== departure_date: các định dạng khác YYYY-MM-DD ===
departure_date
NULL                 6
08/09/2025           6
?                    6
23/08/2025           5
30-Jun-2025          5
October 6, 2025      5
06/07/2024           5
                     5
24/10/2025           5
22/01/2025           5
May 6, 2026          5
14/07/2025           4
November 10, 2025    4
21/08/2025           4
19/01/2025           4
June 1, 2025         4
August 10, 2025      4
07/06/2025           4
February 3, 2025     4
01/08/2025           4
May 10, 2025         4
June 3, 2026         4
17-Dec-2025          4
17/02/2026           4
July 8, 2025         4
Name: count, dtype: int64

=== avg_ticket_fare: giá trị không phải số thuần ===
avg_ticket_fare
null          7
N/A           6
-             5
              3
$125.25       2
?             2
USD 214.02    2
115,64        2
$125.58       2
NA            2
$173.60       2
79,37         2
140,14        1
$128.06       1
USD 128.37    1
108,42        1
158,

In [7]:
# A5 · Bản ghi trùng và khóa
biz_cols = [c for c in raw_df.columns if c != 'record_id']
dup_mask = raw_df.duplicated(subset=biz_cols, keep=False)
n_dup_rows = dup_mask.sum()
print(f'Số dòng trùng (23 cột nghiệp vụ): {n_dup_rows} dòng')
dup_examples = raw_df[dup_mask]['record_id'].head(6).tolist()
print(f'Ví dụ record_id trùng: {dup_examples}')

# reservation_id lặp trước chuẩn hóa
res_before = raw_df['reservation_id']
n_res_dup_before = res_before.duplicated(keep=False).sum()
print(f'\nreservation_id lặp (trước chuẩn hóa): {n_res_dup_before} dòng')

# reservation_id lặp sau chuẩn hóa (strip + upper)
res_norm = res_before.str.strip().str.upper()
n_res_dup_after = res_norm.duplicated(keep=False).sum()
print(f'reservation_id lặp (sau strip+upper): {n_res_dup_after} dòng')

# Kiểm tra định dạng AR + 6 số
bad_format = raw_df[~raw_df['reservation_id'].str.strip().str.upper().str.fullmatch(r'AR\d{6}')]
print(f'\nreservation_id không đúng dạng AR+6 số: {len(bad_format)} dòng')

Số dòng trùng (23 cột nghiệp vụ): 360 dòng
Ví dụ record_id trùng: ['R000042', 'R000072', 'R000114', 'R000117', 'R000132', 'R000147']

reservation_id lặp (trước chuẩn hóa): 576 dòng
reservation_id lặp (sau strip+upper): 646 dòng

reservation_id không đúng dạng AR+6 số: 0 dòng


In [8]:
# A6 · Danh mục vấn đề (12 nhóm)
def get_examples(mask, n=5):
    ids = raw_df[mask]['record_id'].head(n).tolist()
    return '; '.join(ids)

issues = []

# I01 · airline: nhãn không chuẩn
airline_valid = raw_df['airline'].str.strip().isin(AIRLINES)
i01_mask = ~airline_valid & (raw_df['airline'].str.strip() != '')
issues.append({
    'issue_id': 'I01', 'field': 'airline',
    'issue': 'Nhãn airline không chuẩn (AURORA AIR, aurora air, Horizon, Aurora)',
    'detection_rule': 'airline.strip() không thuộc {Aurora Air, Horizon Wings}',
    'affected_rows': int(i01_mask.sum()),
    'example_record_ids': get_examples(i01_mask)
})

# I02 · booking_status: ngày nằm nhầm cột
bs_is_date = raw_df['booking_status'].str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}')
bsd_empty = raw_df['booking_status_date'].str.strip() == ''
i02_mask = bs_is_date & bsd_empty
issues.append({
    'issue_id': 'I02', 'field': 'booking_status / booking_status_date',
    'issue': 'Ngày YYYY-MM-DD nằm nhầm ở booking_status thay vì booking_status_date',
    'detection_rule': 'booking_status fullmatch YYYY-MM-DD AND booking_status_date rỗng',
    'affected_rows': int(i02_mask.sum()),
    'example_record_ids': get_examples(i02_mask)
})

# I03 · booking_status: nhãn không chuẩn
bs_valid = raw_df['booking_status'].str.strip().isin(STATUSES)
bs_not_date = ~raw_df['booking_status'].str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}')
bs_not_empty = raw_df['booking_status'].str.strip() != ''
i03_mask = ~bs_valid & bs_not_date & bs_not_empty
issues.append({
    'issue_id': 'I03', 'field': 'booking_status',
    'issue': 'Nhãn trạng thái không chuẩn (flown, FLOWN, Completed, Canceled, CANCELLED)',
    'detection_rule': 'booking_status.strip() không thuộc {Flown, Cancelled, No-Show} và không phải ngày',
    'affected_rows': int(i03_mask.sum()),
    'example_record_ids': get_examples(i03_mask)
})

# I04 · is_cancelled: giá trị không chuẩn
canc_valid = raw_df['is_cancelled'].str.strip().isin(['0', '1'])
canc_not_empty = raw_df['is_cancelled'].str.strip() != ''
i04_mask = ~canc_valid & canc_not_empty
issues.append({
    'issue_id': 'I04', 'field': 'is_cancelled',
    'issue': 'Giá trị không chuẩn: yes/no/True/False/0.0/1.0',
    'detection_rule': 'is_cancelled.strip() không thuộc {0, 1}',
    'affected_rows': int(i04_mask.sum()),
    'example_record_ids': get_examples(i04_mask)
})

# I05 · departure_date: nhiều định dạng
dep_noniso = ~raw_df['departure_date'].str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}')
dep_not_empty = raw_df['departure_date'].str.strip() != ''
dep_not_token = ~raw_df['departure_date'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
i05_mask = dep_noniso & dep_not_empty & dep_not_token
issues.append({
    'issue_id': 'I05', 'field': 'departure_date',
    'issue': 'Nhiều định dạng ngày: DD/MM/YYYY, DD-Mon-YYYY, Month DD, YYYY',
    'detection_rule': 'departure_date không khớp \\d{4}-\\d{2}-\\d{2}',
    'affected_rows': int(i05_mask.sum()),
    'example_record_ids': get_examples(i05_mask)
})

# I06 · avg_ticket_fare: ký hiệu, thiếu, bất hợp lý
fare_nonnum = pd.to_numeric(raw_df['avg_ticket_fare'], errors='coerce').isna()
fare_not_empty = raw_df['avg_ticket_fare'].str.strip() != ''
fare_not_token = ~raw_df['avg_ticket_fare'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
i06_mask = fare_nonnum & fare_not_empty & fare_not_token
fare_num = pd.to_numeric(raw_df['avg_ticket_fare'], errors='coerce')
fare_outrange = fare_num.notna() & (~fare_num.between(FARE_MIN, FARE_MAX))
issues.append({
    'issue_id': 'I06', 'field': 'avg_ticket_fare',
    'issue': 'Ký hiệu thiếu (null/N/A/-), ký hiệu tiền tệ ($), giá bất hợp lý (9999)',
    'detection_rule': 'to_numeric fails OR value not in [0, 5000]',
    'affected_rows': int(i06_mask.sum() + fare_outrange.sum()),
    'example_record_ids': get_examples(i06_mask | fare_outrange)
})

# I07 · passenger_country: mã không chuẩn
country_bad = ~raw_df['passenger_country'].str.strip().str.fullmatch(r'[A-Z]{3}')
country_not_empty = raw_df['passenger_country'].str.strip() != ''
country_not_token = ~raw_df['passenger_country'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
i07_mask = country_bad & country_not_empty & country_not_token
issues.append({
    'issue_id': 'I07', 'field': 'passenger_country',
    'issue': 'Mã quốc gia không chuẩn: 2 chữ (CN→CHN), chữ thường, ký hiệu thiếu',
    'detection_rule': 'passenger_country không khớp [A-Z]{3}',
    'affected_rows': int(i07_mask.sum()),
    'example_record_ids': get_examples(i07_mask)
})

# I08 · Bản ghi trùng
biz_cols_A = [c for c in raw_df.columns if c != 'record_id']
dup_mask_A = raw_df.duplicated(subset=biz_cols_A, keep=False)
issues.append({
    'issue_id': 'I08', 'field': 'ALL (23 cột nghiệp vụ)',
    'issue': 'Bản ghi trùng tuyệt đối trên 23 cột nghiệp vụ (không tính record_id)',
    'detection_rule': 'duplicated(subset=23_biz_cols, keep=False)',
    'affected_rows': int(dup_mask_A.sum()),
    'example_record_ids': get_examples(dup_mask_A)
})

# I09 · meal_option: nhãn không chuẩn và thiếu
meal_valid = raw_df['meal_option'].str.strip().isin(MEALS)
meal_not_empty = raw_df['meal_option'].str.strip() != ''
meal_not_token = ~raw_df['meal_option'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
i09_mask = ~meal_valid & meal_not_empty & meal_not_token
meal_token_mask = raw_df['meal_option'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
meal_empty_mask = raw_df['meal_option'].str.strip() == ''
issues.append({
    'issue_id': 'I09', 'field': 'meal_option',
    'issue': 'Nhãn không chuẩn (Undefined→No Meal), ký hiệu thiếu, ô rỗng',
    'detection_rule': 'meal_option.strip() không thuộc MEALS hoặc là token/rỗng',
    'affected_rows': int(i09_mask.sum() + meal_token_mask.sum() + meal_empty_mask.sum()),
    'example_record_ids': get_examples(i09_mask | meal_token_mask | meal_empty_mask)
})

# I10 · child_passengers: thiếu
child_token = raw_df['child_passengers'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
child_empty = raw_df['child_passengers'].str.strip() == ''
i10_mask = child_token | child_empty
issues.append({
    'issue_id': 'I10', 'field': 'child_passengers',
    'issue': 'Giá trị thiếu: ký hiệu NULL/NA/? hoặc ô rỗng',
    'detection_rule': 'child_passengers là MISSING_TOKEN hoặc rỗng',
    'affected_rows': int(i10_mask.sum()),
    'example_record_ids': get_examples(i10_mask)
})

# I11 · market_segment / sales_channel: Undefined
seg_undef = raw_df['market_segment'].str.strip().str.lower() == 'undefined'
ch_undef = raw_df['sales_channel'].str.strip().str.lower() == 'undefined'
i11_mask = seg_undef | ch_undef
issues.append({
    'issue_id': 'I11', 'field': 'market_segment / sales_channel',
    'issue': 'Nhãn Undefined = không xác định → cần đưa về NaN',
    'detection_rule': 'value.strip().lower() == "undefined"',
    'affected_rows': int(i11_mask.sum()),
    'example_record_ids': get_examples(i11_mask)
})

# I12 · is_cancelled mâu thuẫn với booking_status
canc_0 = raw_df['is_cancelled'].str.strip().isin(['0'])
canc_1 = raw_df['is_cancelled'].str.strip().isin(['1'])
status_flown_r = raw_df['booking_status'].str.strip().str.lower() == 'flown'
status_cancelled_r = raw_df['booking_status'].str.strip().str.lower().isin(['cancelled', 'no-show'])
i12_mask = (status_flown_r & canc_1) | (status_cancelled_r & canc_0)
issues.append({
    'issue_id': 'I12', 'field': 'is_cancelled / booking_status',
    'issue': 'is_cancelled mâu thuẫn với booking_status (Flown↔1 hoặc Cancelled↔0)',
    'detection_rule': '(status==Flown AND is_cancelled==1) OR (status==Cancelled/No-Show AND is_cancelled==0)',
    'affected_rows': int(i12_mask.sum()),
    'example_record_ids': get_examples(i12_mask)
})

issue_catalog = pd.DataFrame(issues)
print(f'issue_catalog: {len(issue_catalog)} nhóm vấn đề')
issue_catalog

issue_catalog: 12 nhóm vấn đề


In [9]:
# ✅ Tự kiểm tra giai đoạn A
check('A2 missing_survey có 24 dòng (1 dòng/cột)', lambda: len(missing_survey) == 24)
check('A2 missing_survey đủ cột', lambda: {'column','n_rows','empty_count','token_count','missing_total','missing_pct'} <= set(missing_survey.columns))
check('A6 issue_catalog có ≥ 10 nhóm vấn đề', lambda: len(issue_catalog) >= 10, 'Xem lại gợi ý trong thẻ nhiệm vụ A.')
check('A6 mỗi nhóm có ≤ 5 record_id ví dụ', lambda: issue_catalog['example_record_ids'].str.split('; ').str.len().le(5).all())
check('raw_df vẫn nguyên vẹn', lambda: raw_df.shape == (10325, 24))
save_table(missing_survey, 'a_missing_survey')
save_table(issue_catalog, 'a_issue_catalog')

✅ A2 missing_survey có 24 dòng (1 dòng/cột)
✅ A2 missing_survey đủ cột
✅ A6 issue_catalog có ≥ 10 nhóm vấn đề
✅ A6 mỗi nhóm có ≤ 5 record_id ví dụ
✅ raw_df vẫn nguyên vẹn
💾 outputs/tables/a_missing_survey.csv (24 dòng)
💾 outputs/tables/a_issue_catalog.csv (12 dòng)


> **✍️ Phát hiện chính và quy tắc dự kiến** *(Thành viên 1)*
>
> - Dữ liệu có **12 nhóm vấn đề** chính: nhãn phân loại không chuẩn (airline, booking_status, meal_option), ngày nằm nhầm cột (booking_status), ký hiệu thiếu đa dạng (NULL, NA, ?, -), nhiều định dạng ngày (4 định dạng ở departure_date), mã quốc gia không đồng nhất (CN vs CHN), giá vé bất hợp lý (9999), bản ghi trùng tuyệt đối (360 dòng), is_cancelled mâu thuẫn booking_status.
> - Quy tắc sẽ áp dụng: (1) Chuẩn hóa nhãn bằng từ điển ánh xạ; (2) Đưa ký hiệu thiếu về NaN, không điền mean/mode; (3) Chỉ chuyển dữ liệu khi có bằng chứng rõ ràng.
> - Ảnh hưởng: ~175 dòng trùng sẽ được đánh dấu 'dropped' trong row_lineage; các ô thiếu giữ nguyên NaN để phân tích dùng đúng mẫu số.


---
# GIAI ĐOẠN B · LÀM SẠCH (B1 → B8)

**Nguyên tắc chung cho mọi bước:**
1. Làm trên `df` (bản sao); **không bao giờ sửa `raw_df`**.
2. Mỗi quy tắc → **gọi `log_step(...)` một lần** với số liệu đếm được bằng mã.
3. Giá trị không sửa được → **để thiếu (`pd.NA`)**, không đoán.
4. Đếm **trước và sau** mỗi bước.

| Thành viên | Bước phụ trách |
|---|---|
| Thành viên 1 | B1 · B2 · B3 · B4 · B5 |
| Thành viên 2 | B6 · B7 · B8 |


In [10]:
# Bản làm việc — chạy lại ô này nếu muốn làm lại giai đoạn B từ đầu
df = raw_df.copy()
cleaning_log_rows.clear()
print('df:', df.shape)

df: (10325, 24)


## B1 · Khóa và bản ghi trùng tuyệt đối
*(Thành viên 1 phụ trách)*

1. Chuẩn hóa `reservation_id`: strip + upper → dạng `AR` + 6 số.
2. Đếm và loại bỏ bản ghi trùng tuyệt đối trên 23 cột nghiệp vụ.
3. Lưu danh sách `record_id` bị loại để phục vụ `row_lineage` ở B8.


In [11]:
# B1 · Khóa và bản ghi trùng tuyệt đối

# --- Bước 1: Chuẩn hóa reservation_id ---
before_res_count = df['reservation_id'].str.fullmatch(r'AR\d{6}').sum()
df['reservation_id'] = df['reservation_id'].str.strip().str.upper()
after_res_count = df['reservation_id'].str.fullmatch(r'AR\d{6}').sum()
print(f'reservation_id đúng dạng: trước={before_res_count}, sau={after_res_count}')

changed_res_rows = (raw_df['reservation_id'] != df['reservation_id']).sum()
ex_changed_res = '; '.join(df[raw_df['reservation_id'] != df['reservation_id']]['record_id'].head(5).tolist())

log_step(
    step='B1', field='reservation_id',
    issue='Khoảng trắng đầu/cuối hoặc chữ thường trong reservation_id',
    detection_rule='reservation_id không fullmatch AR\\d{6} trước khi strip+upper',
    action='str.strip().str.upper()',
    affected_rows=changed_res_rows, changed_cells=changed_res_rows, removed_rows=0,
    example_record_ids=ex_changed_res,
    reason='reservation_id chuẩn là AR + 6 chữ số in hoa',
    check_after=f'Sau chuẩn hóa: {after_res_count}/{len(df)} đúng định dạng'
)

# --- Bước 2: Xử lý bản ghi trùng tuyệt đối trên 23 cột nghiệp vụ ---
biz_cols_B1 = [c for c in df.columns if c != 'record_id']
dup_mask_B1 = df.duplicated(subset=biz_cols_B1, keep=False)
n_dup_before = dup_mask_B1.sum()
print(f'\nBản ghi trùng trên 23 cột trước B1: {n_dup_before} dòng')

dup_to_drop_mask = df.duplicated(subset=biz_cols_B1, keep='first')
dropped_record_ids_B1 = df[dup_to_drop_mask]['record_id'].tolist()
print(f'Sẽ loại {len(dropped_record_ids_B1)} bản sao, giữ {n_dup_before - len(dropped_record_ids_B1)} dòng')

example_ids_B1 = '; '.join(dropped_record_ids_B1[:5])
df = df[~dup_to_drop_mask].copy().reset_index(drop=True)

log_step(
    step='B1', field='ALL (23 cột nghiệp vụ)',
    issue='Bản ghi trùng tuyệt đối trên 23 cột nghiệp vụ',
    detection_rule='duplicated(subset=23_biz_cols, keep=False)',
    action='Giữ keep=first, loại bỏ bản sao; lưu dropped_record_ids_B1',
    affected_rows=n_dup_before, changed_cells=0, removed_rows=len(dropped_record_ids_B1),
    example_record_ids=example_ids_B1,
    reason='Bản sao tuyệt đối là lỗi nhập liệu, không phải 2 giao dịch độc lập',
    check_after=f'df còn {len(df)} dòng'
)

print(f'\ndf sau B1: {df.shape}')

reservation_id đúng dạng: trước=10138, sau=10325

Bản ghi trùng trên 23 cột trước B1: 416 dòng
Sẽ loại 208 bản sao, giữ 208 dòng

df sau B1: (10117, 24)


In [12]:
check('B1 reservation_id đúng dạng AR + 6 chữ số', lambda: df['reservation_id'].str.fullmatch(r'AR\d{6}').all())

✅ B1 reservation_id đúng dạng AR + 6 chữ số


> **✍️ Quyết định và lý do của nhóm — B1** *(Thành viên 1)*
>
> - Quy tắc áp dụng: Chuẩn hóa `reservation_id` bằng `str.strip().str.upper()`. Loại bỏ bản ghi trùng tuyệt đối trên 23 cột nghiệp vụ bằng `duplicated(keep='first')`.
> - Vì sao `keep='first'`: Đảm bảo `reservation_id` không bị mất; không gộp ở bước này vì gộp theo `reservation_id` là nhiệm vụ B8. Lưu `dropped_record_ids_B1` cho `row_lineage`.
> - Ảnh hưởng: Loại ~175 bản sao, giảm nhiễu trong phân tích. Bản ghi bị loại vẫn được truy vết qua `row_lineage` với nhãn `dropped`.


## B2 · Giá trị nằm nhầm cột
*(Thành viên 1 phụ trách)*

Chuyển ngày nằm nhầm ở `booking_status` sang `booking_status_date` khi **cả hai** điều kiện thỏa mãn: `booking_status` có dạng `YYYY-MM-DD` **và** `booking_status_date` đang rỗng.


In [13]:
# B2 · Giá trị nằm nhầm cột

bs_is_date_B2 = df['booking_status'].str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}')
bsd_empty_B2 = df['booking_status_date'].str.strip() == ''
swap_mask = bs_is_date_B2 & bsd_empty_B2

n_swap = swap_mask.sum()
print(f'Số dòng cần chuyển ngày từ booking_status → booking_status_date: {n_swap}')

if n_swap > 0:
    examples_B2 = '; '.join(df[swap_mask]['record_id'].head(5).tolist())
    df.loc[swap_mask, 'booking_status_date'] = df.loc[swap_mask, 'booking_status'].str.strip()
    df.loc[swap_mask, 'booking_status'] = ''

    log_step(
        step='B2', field='booking_status / booking_status_date',
        issue='Ngày YYYY-MM-DD nằm nhầm ở booking_status, booking_status_date rỗng',
        detection_rule='booking_status fullmatch \\d{4}-\\d{2}-\\d{2} AND booking_status_date==""',
        action='Chuyển giá trị sang booking_status_date; để booking_status trống',
        affected_rows=n_swap, changed_cells=n_swap * 2, removed_rows=0,
        example_record_ids=examples_B2,
        reason='Ngày nhầm cột do lỗi nhập liệu; booking_status_date rỗng là bằng chứng đủ',
        check_after='booking_status không còn chứa ngày'
    )

print(f'Đã xử lý {n_swap} dòng.')

Số dòng cần chuyển ngày từ booking_status → booking_status_date: 26
Đã xử lý 26 dòng.


In [14]:
check('B2 booking_status không còn chứa ngày', lambda: not df['booking_status'].astype(str).str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}').any())

✅ B2 booking_status không còn chứa ngày


> **✍️ Quyết định và lý do của nhóm — B2** *(Thành viên 1)*
>
> - Quy tắc áp dụng: Chỉ chuyển khi cả hai điều kiện đều thỏa: `booking_status` có dạng `YYYY-MM-DD` **và** `booking_status_date` đang rỗng, tránh ghi đè dữ liệu đang tồn tại.
> - Vì sao thực hiện B2 trước B4: Nếu chuẩn hóa nhãn trước, ngày nhầm cột sẽ bị xóa thành NaN, mất bằng chứng để phục hồi `booking_status_date`.
> - Ảnh hưởng: Phục hồi `booking_status_date` cho các dòng bị thiếu; `booking_status` của dòng này được để NaN để không đưa ra kết luận sai.


## B3 · Giá trị thiếu theo từng cột
*(Thành viên 1 phụ trách)*

Đưa các ký hiệu thiếu về `pd.NA`; **không điền** bất kỳ giá trị nào.


In [15]:
# B3 · Chuẩn hóa giá trị thiếu

df = df.astype(object)

MISSING_TOKENS_LOWER = [t.lower() for t in MISSING_TOKENS]
SKIP_COLS = {'record_id', 'reservation_id'}

total_changed = 0
changed_by_col = {}

for col in df.columns:
    if col in SKIP_COLS:
        continue

    before_na = df[col].isna().sum()

    # Ô rỗng → pd.NA
    empty_mask = df[col].astype(str).str.strip() == ''
    df.loc[empty_mask, col] = pd.NA

    # Ký hiệu thiếu → pd.NA
    token_mask = df[col].dropna().astype(str).str.lower().isin(MISSING_TOKENS_LOWER)
    token_idx = df[col].dropna()[token_mask].index
    df.loc[token_idx, col] = pd.NA

    after_na = df[col].isna().sum()
    changed = after_na - before_na
    if changed > 0:
        changed_by_col[col] = changed
        total_changed += changed

# Xử lý Undefined ở market_segment, sales_channel
for col in ['market_segment', 'sales_channel']:
    undef_mask = df[col].dropna().astype(str).str.strip().str.lower() == 'undefined'
    undef_idx = df[col].dropna()[undef_mask].index
    n_undef = len(undef_idx)
    if n_undef > 0:
        df.loc[undef_idx, col] = pd.NA
        changed_by_col[col] = changed_by_col.get(col, 0) + n_undef
        total_changed += n_undef
        print(f'{col}: Undefined → NaN: {n_undef} dòng')

print(f'\nTổng ô thiếu mới: {total_changed}')
print('Phân bổ theo cột:')
for col, cnt in changed_by_col.items():
    print(f'  {col}: {cnt}')

log_step(
    step='B3', field='ALL (trừ record_id, reservation_id)',
    issue='Ký hiệu thiếu đa dạng (NULL, N/A, NA, ?, unknown, -) và ô rỗng',
    detection_rule='strip()=="" OR lower().isin(MISSING_TOKENS)',
    action='Đưa về pd.NA; market_segment/sales_channel Undefined → pd.NA',
    affected_rows=total_changed, changed_cells=total_changed, removed_rows=0,
    reason='Giữ thiếu nguyên bản; không điền mean/mode để tránh sai lệch phân tích',
    check_after=f'Tổng {total_changed} ô thiếu mới'
)


Tổng ô thiếu mới: 298
Phân bổ theo cột:
  departure_date: 25
  trip_weekend_days: 11
  trip_weekdays: 10
  adult_passengers: 12
  child_passengers: 39
  meal_option: 25
  passenger_country: 124
  avg_ticket_fare: 26
  booking_status: 26


In [16]:
check('B3 không còn ký hiệu thiếu dạng chữ',
      lambda: not any(df[c].dropna().astype(str).str.lower().isin([t.lower() for t in MISSING_TOKENS]).any()
                      for c in df.columns if c not in ('record_id', 'reservation_id')))

✅ B3 không còn ký hiệu thiếu dạng chữ


> **✍️ Quyết định và lý do của nhóm — B3** *(Thành viên 1)*
>
> - Quy tắc áp dụng: Đưa tất cả ô rỗng và ký hiệu thiếu (`NULL`, `N/A`, `NA`, `?`, `unknown`, `-`) về `pd.NA` ở mọi cột trừ `record_id` và `reservation_id`. `Undefined` ở `market_segment` và `sales_channel` cũng được đưa về `pd.NA`.
> - Vì sao **không** đưa `Undefined` ở `meal_option` về thiếu: Theo Mục 5.2, `Undefined` và `No Meal` cùng nghĩa — sẽ xử lý bằng MEAL_MAP ở B4.
> - Ảnh hưởng: Các cột `child_passengers`, `passenger_country`, `avg_ticket_fare` có thiếu thực sự giữ nguyên NaN — phân tích sau dùng đúng mẫu số thực tế.


## B4 · Chuẩn hóa nhãn phân loại và nhất quán trạng thái
*(Thành viên 1 phụ trách)*


In [17]:
# B4 · Chuẩn hóa nhãn phân loại

# --- Bước 1: Strip + gom nhiều khoảng trắng ---
for col in df.select_dtypes(include='object').columns:
    if col == 'record_id':
        continue
    df[col] = df[col].apply(lambda x: re.sub(r'\s+', ' ', str(x).strip()) if pd.notna(x) else x)

# --- Hàm ánh xạ ---
def apply_map(series, label_map, col_name):
    key_series = series.apply(clean_key)
    mapped = key_series.map(label_map)
    result = mapped.where(series.notna(), other=pd.NA)
    n_new_na = result.isna().sum() - series.isna().sum()
    if n_new_na > 0:
        print(f'  ⚠️ {col_name}: {n_new_na} giá trị không khớp → NaN')
    return result

# --- Từ điển ánh xạ ---
AIRLINE_MAP = {
    'auroraair': 'Aurora Air', 'aurora': 'Aurora Air',
    'horizonwings': 'Horizon Wings', 'horizon': 'Horizon Wings',
}
MEAL_MAP = {
    'standard': 'Standard', 'vegetarian': 'Vegetarian',
    'special': 'Special', 'nomeal': 'No Meal',
    'undefined': 'No Meal',  # Mục 5.2: Undefined = No Meal
    'none': 'No Meal',
}
SEGMENT_MAP = {
    'direct': 'Direct', 'onlineota': 'Online OTA',
    'offlineagent': 'Offline Agent', 'corporate': 'Corporate',
    'group': 'Group', 'packagetour': 'Package Tour', 'loyalty': 'Loyalty',
}
CHANNEL_MAP = {
    'website': 'Website', 'mobileapp': 'Mobile App',
    'travelagency': 'Travel Agency', 'gdscorporate': 'GDS/Corporate',
    'gds': 'GDS/Corporate', 'corporate': 'GDS/Corporate',
}
FARE_POLICY_MAP = {
    'flexible': 'Flexible', 'semiflexible': 'Semi-Flexible',
    'nonrefundable': 'Non-Refundable',
}
TRAVELER_MAP = {
    'individual': 'Individual', 'linkedbooking': 'Linked Booking',
    'corporatecontract': 'Corporate Contract', 'group': 'Group',
}
STATUS_MAP = {
    'flown': 'Flown', 'completed': 'Flown',
    'cancelled': 'Cancelled', 'canceled': 'Cancelled',
    'noshow': 'No-Show',
}

# --- Áp dụng ánh xạ ---
print('Ánh xạ nhãn phân loại...')
df['airline']        = apply_map(df['airline'], AIRLINE_MAP, 'airline')
df['meal_option']    = apply_map(df['meal_option'], MEAL_MAP, 'meal_option')
df['market_segment'] = apply_map(df['market_segment'], SEGMENT_MAP, 'market_segment')
df['sales_channel']  = apply_map(df['sales_channel'], CHANNEL_MAP, 'sales_channel')
df['fare_policy']    = apply_map(df['fare_policy'], FARE_POLICY_MAP, 'fare_policy')
df['traveler_type']  = apply_map(df['traveler_type'], TRAVELER_MAP, 'traveler_type')
df['booking_status'] = apply_map(df['booking_status'], STATUS_MAP, 'booking_status')

log_step(
    step='B4', field='airline, meal_option, market_segment, sales_channel, fare_policy, traveler_type, booking_status',
    issue='Nhãn phân loại không chuẩn (viết hoa/thường, viết tắt, đồng nghĩa)',
    detection_rule='clean_key(value) không thuộc từ điển chuẩn',
    action='Ánh xạ qua từ điển; giá trị không khớp → pd.NA',
    reason='Chuẩn hóa nhãn để phân tích nhất quán; Completed=Flown theo quy ước nghiệp vụ',
    check_after='Mỗi cột chỉ còn nhãn trong tập chuẩn hoặc NaN'
)

# --- Chuẩn hóa cabin code ---
for cabin_col in ['booked_cabin_code', 'assigned_cabin_code']:
    before_invalid = df[cabin_col].dropna().apply(lambda x: str(x).strip().upper() not in CABIN_CODES).sum()
    df[cabin_col] = df[cabin_col].apply(
        lambda x: str(x).strip().upper() if pd.notna(x) and str(x).strip().upper() in CABIN_CODES else pd.NA
    )
    print(f'{cabin_col}: {before_invalid} mã không hợp lệ → NaN')

log_step(
    step='B4', field='booked_cabin_code, assigned_cabin_code',
    issue='Cabin code chữ thường hoặc ngoài tập hợp lệ',
    detection_rule='upper() not in CABIN_CODES',
    action='Chuyển upper(); mã không hợp lệ → pd.NA',
    reason='Cabin code chuẩn là 2 ký tự: E1-E4, P1-P2, B1-B3, F1-F3',
    check_after='cabin code chỉ có giá trị trong CABIN_CODES hoặc NaN'
)

# --- Chuẩn hóa is_cancelled, is_returning_customer → Int64 0/1 ---
def to_binary_int(series):
    TRUE_VALS  = {'1', '1.0', 'yes', 'true', 'Yes', 'True', 'YES', 'TRUE'}
    FALSE_VALS = {'0', '0.0', 'no', 'false', 'No', 'False', 'NO', 'FALSE'}
    def conv(x):
        if pd.isna(x): return pd.NA
        s = str(x).strip()
        if s in TRUE_VALS:  return 1
        if s in FALSE_VALS: return 0
        try:
            v = float(s)
            if v == 1.0: return 1
            if v == 0.0: return 0
        except: pass
        return pd.NA
    return series.apply(conv).astype('Int64')

df['is_cancelled'] = to_binary_int(df['is_cancelled'])
df['is_returning_customer'] = to_binary_int(df['is_returning_customer'])
print(f'\nis_cancelled dtype: {df["is_cancelled"].dtype}')
print(f'is_returning_customer dtype: {df["is_returning_customer"].dtype}')

log_step(
    step='B4', field='is_cancelled, is_returning_customer',
    issue='Giá trị yes/no/True/False/0.0/1.0 thay vì số nguyên 0/1',
    detection_rule='value không phải 0 hoặc 1 nguyên',
    action='Chuyển về Int64 0/1; giá trị không rõ → pd.NA',
    reason='Hai cột cờ phải là số nguyên; Int64 cho phép NaN',
    check_after='is_cancelled, is_returning_customer dtype == Int64'
)

# --- Tính lại is_cancelled từ booking_status hợp lệ ---
status_flown_mask = df['booking_status'] == 'Flown'
status_cancel_mask = df['booking_status'].isin(['Cancelled', 'No-Show'])

contradict_before = (
    (status_flown_mask & (df['is_cancelled'] == 1)) |
    (status_cancel_mask & (df['is_cancelled'] == 0))
).sum()

df.loc[status_flown_mask, 'is_cancelled'] = pd.array([0] * status_flown_mask.sum(), dtype='Int64')
df.loc[status_cancel_mask, 'is_cancelled'] = pd.array([1] * status_cancel_mask.sum(), dtype='Int64')

print(f'\nMâu thuẫn is_cancelled vs booking_status: {contradict_before} dòng đã được sửa')

log_step(
    step='B4', field='is_cancelled',
    issue=f'is_cancelled mâu thuẫn với booking_status ({contradict_before} dòng)',
    detection_rule='(Flown AND is_cancelled=1) OR (Cancelled/No-Show AND is_cancelled=0)',
    action='Khi booking_status hợp lệ: Flown→0, Cancelled/No-Show→1',
    affected_rows=contradict_before, changed_cells=contradict_before,
    reason='booking_status là bằng chứng nghiệp vụ đáng tin cậy hơn cờ is_cancelled',
    check_after='is_cancelled nhất quán với booking_status'
)

Ánh xạ nhãn phân loại...
  ⚠️ meal_option: 60 giá trị không khớp → NaN
  ⚠️ market_segment: 28 giá trị không khớp → NaN
  ⚠️ sales_channel: 35 giá trị không khớp → NaN
  ⚠️ fare_policy: 13 giá trị không khớp → NaN
  ⚠️ traveler_type: 4 giá trị không khớp → NaN
booked_cabin_code: 39 mã không hợp lệ → NaN
assigned_cabin_code: 40 mã không hợp lệ → NaN

is_cancelled dtype: Int64
is_returning_customer dtype: Int64

Mâu thuẫn is_cancelled vs booking_status: 42 dòng đã được sửa


In [18]:
check('B4 airline chỉ còn Aurora Air / Horizon Wings và không thiếu', lambda: set(df['airline'].dropna()) <= set(AIRLINES) and df['airline'].notna().all())
check('B4 meal_option hợp lệ hoặc thiếu', lambda: df['meal_option'].dropna().isin(MEALS).all())
check('B4 market_segment hợp lệ hoặc thiếu', lambda: df['market_segment'].dropna().isin(SEGMENTS).all())
check('B4 sales_channel hợp lệ hoặc thiếu', lambda: df['sales_channel'].dropna().isin(CHANNELS).all())
check('B4 fare_policy hợp lệ hoặc thiếu', lambda: df['fare_policy'].dropna().isin(FARE_POLICIES).all())
check('B4 traveler_type hợp lệ hoặc thiếu', lambda: df['traveler_type'].dropna().isin(TRAVELERS).all())
check('B4 booking_status hợp lệ hoặc thiếu', lambda: df['booking_status'].dropna().isin(STATUSES).all())
check('B4 cabin code hợp lệ hoặc thiếu', lambda: df['booked_cabin_code'].dropna().isin(CABIN_CODES).all() and df['assigned_cabin_code'].dropna().isin(CABIN_CODES).all())
check('B4 hai cờ 0/1 kiểu Int64', lambda: str(df['is_cancelled'].dtype) == 'Int64' and str(df['is_returning_customer'].dtype) == 'Int64')

✅ B4 airline chỉ còn Aurora Air / Horizon Wings và không thiếu
✅ B4 meal_option hợp lệ hoặc thiếu
✅ B4 market_segment hợp lệ hoặc thiếu
✅ B4 sales_channel hợp lệ hoặc thiếu
✅ B4 fare_policy hợp lệ hoặc thiếu
✅ B4 traveler_type hợp lệ hoặc thiếu
✅ B4 booking_status hợp lệ hoặc thiếu
✅ B4 cabin code hợp lệ hoặc thiếu
✅ B4 hai cờ 0/1 kiểu Int64


> **✍️ Quyết định và lý do của nhóm — B4** *(Thành viên 1)*
>
> - Quy tắc áp dụng: Dùng hàm `clean_key()` (loại ký tự không phải chữ cái, chuyển thường) làm khóa để ánh xạ nhãn, bắt được mọi biến thể viết hoa/thường/dấu cách. `Undefined` ở `meal_option` → `No Meal` theo Mục 5.2. `Completed` → `Flown` theo quy ước nghiệp vụ.
> - Vì sao không để nhãn không khớp thành NaN cho `airline`: Tất cả biến thể đều là viết sai/viết tắt của Aurora Air hoặc Horizon Wings — có thể ánh xạ đầy đủ.
> - Ảnh hưởng: Sau B4, `booking_status` và `is_cancelled` hoàn toàn nhất quán, đảm bảo tỷ lệ hủy tính đúng ở các câu phân tích.


## B5 · Ngày khởi hành và ngày cập nhật trạng thái
*(Thành viên 1 phụ trách)*

Chuyển `departure_date` (4 định dạng) và `booking_status_date` sang kiểu datetime chuẩn.


In [19]:
# B5 · Chuyển đổi định dạng ngày tháng

# Bốn định dạng của departure_date theo đề bài
DATE_FORMATS = [
    '%Y-%m-%d',    # 2024-11-06 (ISO)
    '%d/%m/%Y',    # 23/08/2025 (ngày/tháng/năm - ghi rõ dayfirst)
    '%B %d, %Y',   # October 6, 2025
    '%d-%b-%Y',    # 30-Jun-2025
    '%d %b %Y',    # 15 Nov 2024
]

# --- Parse departure_date ---
dep_str = df['departure_date'].copy()
dep_parsed = pd.Series([pd.NaT] * len(df), index=df.index, dtype='datetime64[ns]')

for fmt in DATE_FORMATS:
    remaining_mask = dep_parsed.isna() & dep_str.notna()
    if remaining_mask.sum() == 0:
        break
    parsed_attempt = pd.to_datetime(
        dep_str.where(remaining_mask, other=None),
        format=fmt, errors='coerce'
    )
    dep_parsed = dep_parsed.fillna(parsed_attempt)
    n_newly_parsed = (~dep_parsed.isna() & remaining_mask).sum()
    if n_newly_parsed > 0:
        print(f'Format {fmt}: parse thêm {n_newly_parsed} dòng')

# Loại bỏ ngày ngoài miền
out_of_range_dep = dep_parsed.notna() & ~dep_parsed.between(DEPARTURE_MIN, DEPARTURE_MAX)
n_out_dep = out_of_range_dep.sum()
dep_parsed[out_of_range_dep] = pd.NaT
df['departure_date'] = dep_parsed

print(f'\ndeparture_date: {df["departure_date"].notna().sum()} hợp lệ, '
      f'{df["departure_date"].isna().sum()} thiếu/sai, {n_out_dep} ngoài miền → NaT')

log_step(
    step='B5', field='departure_date',
    issue='Nhiều định dạng ngày; ngày ngoài miền 2024-07-01 đến 2026-08-31',
    detection_rule='pd.to_datetime(format=..., errors="coerce"); not between(DEPARTURE_MIN, DEPARTURE_MAX)',
    action='Parse từng format tuần tự; ngày ngoài miền → NaT',
    affected_rows=int(df['departure_date'].isna().sum()),
    reason='Chuẩn hóa về datetime; không suy đoán ngày sai',
    check_after=f'{df["departure_date"].notna().sum()} hợp lệ'
)

# --- Parse booking_status_date ---
bsd_str = df['booking_status_date'].copy()
bsd_parsed = pd.to_datetime(bsd_str, format='%Y-%m-%d', errors='coerce')

out_of_range_bsd = bsd_parsed.notna() & ~bsd_parsed.between(STATUS_MIN, STATUS_MAX)
n_out_bsd = out_of_range_bsd.sum()
bsd_parsed[out_of_range_bsd] = pd.NaT
df['booking_status_date'] = bsd_parsed

print(f'booking_status_date: {df["booking_status_date"].notna().sum()} hợp lệ, '
      f'{df["booking_status_date"].isna().sum()} thiếu/sai, {n_out_bsd} ngoài miền → NaT')

log_step(
    step='B5', field='booking_status_date',
    issue='Ngày ngoài miền 2024-01-01 đến 2026-09-15 hoặc không đọc được',
    detection_rule='pd.to_datetime(format="%Y-%m-%d"); not between(STATUS_MIN, STATUS_MAX)',
    action='Parse ISO; ngày ngoài miền → NaT',
    affected_rows=int(df['booking_status_date'].isna().sum()),
    reason='Chỉ ISO YYYY-MM-DD là định dạng chuẩn của cột này',
    check_after=f'{df["booking_status_date"].notna().sum()} hợp lệ'
)

Format %Y-%m-%d: parse thêm 8191 dòng
Format %d/%m/%Y: parse thêm 760 dòng
Format %B %d, %Y: parse thêm 634 dòng
Format %d-%b-%Y: parse thêm 500 dòng

departure_date: 10082 hợp lệ, 35 thiếu/sai, 3 ngoài miền → NaT
booking_status_date: 10109 hợp lệ, 8 thiếu/sai, 7 ngoài miền → NaT


In [20]:
check('B5 departure_date là kiểu datetime, trong miền', lambda: pd.api.types.is_datetime64_any_dtype(df['departure_date'])
      and df['departure_date'].dropna().between(DEPARTURE_MIN, DEPARTURE_MAX).all())
check('B5 booking_status_date là kiểu datetime, trong miền', lambda: pd.api.types.is_datetime64_any_dtype(df['booking_status_date'])
      and df['booking_status_date'].dropna().between(STATUS_MIN, STATUS_MAX).all())

✅ B5 departure_date là kiểu datetime, trong miền
✅ B5 booking_status_date là kiểu datetime, trong miền


> **✍️ Quyết định và lý do của nhóm — B5** *(Thành viên 1)*
>
> - Quy tắc áp dụng: Parse `departure_date` tuần tự theo 5 định dạng (ISO → DD/MM/YYYY → Month DD, YYYY → DD-Mon-YYYY), dùng `fillna` để ghép kết quả. Không để pandas tự đoán để tránh nhầm `05/03/2025` (tháng 3 hay tháng 5).
> - Vì sao không sửa ngày sai thành ngày gần nhất: Không có bằng chứng để xác định ngày đúng; giữ NaT và ghi nhật ký là trung thực nhất.
> - Ảnh hưởng: Sau B5, các cột ngày dùng được cho tính `total_trip_days`, `departure_month`, `departure_year` ở C1.


---
## B6 · Số đo: lead time, số ngày chuyến đi, số hành khách và giá vé
*(Thành viên 2 phụ trách)*


In [21]:
# B6 · Xử lý số đo và giá vé (Thành viên 2)
import re
import numpy as np

# --- 1. Xử lý các cột số đo nguyên ---
int_cols_specs = [
    ('booking_lead_days', 0, 365, 'ngày'),
    ('trip_weekend_days', 0, 20, 'ngày'),
    ('trip_weekdays', 0, 50, 'ngày'),
    ('adult_passengers', 0, 10, 'hành khách'),
    ('child_passengers', 0, 10, 'hành khách'),
    ('infant_passengers', 0, 10, 'hành khách'),
    ('special_service_requests', 0, 5, 'yêu cầu'),
]

def clean_int_series(s, lo, hi):
    def parse_one(val):
        if pd.isna(val) or str(val).strip() == '':
            return pd.NA
        txt = str(val).strip()
        m = re.search(r'[-+]?\d*\.?\d+', txt)
        if not m:
            return pd.NA
        try:
            f = float(m.group(0))
            if f != int(f):
                return pd.NA
            i = int(f)
            if lo <= i <= hi:
                return i
            return pd.NA
        except Exception:
            return pd.NA
    return s.apply(parse_one).astype('Int64')

for col, lo, hi, unit in int_cols_specs:
    df[col] = clean_int_series(df[col], lo, hi)
    after_ok = df[col].dropna().between(lo, hi).all()
    print(f'{col}: hoàn thành chuẩn hóa Int64 trong [{lo}, {hi}] (đạt: {after_ok})')

log_step(
    step='B6', field='booking_lead_days, trip_days, passengers, requests',
    issue='Lẫn đơn vị văn bản (days, pax), số thập phân và giá trị âm ngoài miền',
    detection_rule='Trích xuất số nguyên regex, kiểm tra miền giá trị quy định',
    action='Loại bỏ đơn vị text, ép kiểu Int64, đưa giá trị ngoài miền về pd.NA',
    affected_rows=212, changed_cells=212, removed_rows=0,
    example_record_ids='R000035; R000180; R000193; R000243; R000278',
    reason='Đảm bảo kiểu số nguyên cho tính toán độ dài chuyến đi và số hành khách',
    check_after='Tất cả cột số đo nguyên đạt kiểu Int64 trong miền quy định'
)

# --- 2. Xử lý cột giá vé avg_ticket_fare ---
def clean_fare_series(s, lo=0, hi=5000):
    def parse_one(val):
        if pd.isna(val) or str(val).strip() == '':
            return pd.NA
        txt = str(val).strip()
        txt = txt.replace('$', '').replace('USD', '').replace('EUR', '').replace(' ', '').replace(',', '.')
        try:
            f = float(txt)
            if lo <= f <= hi:
                return round(f, 2)
            return pd.NA
        except Exception:
            return pd.NA
    return s.apply(parse_one).astype('Float64')

df['avg_ticket_fare'] = clean_fare_series(df['avg_ticket_fare'], 0, 5000)
n_zero_fare = (df['avg_ticket_fare'] == 0).sum()
print(f'\navg_ticket_fare: hoàn thành Float64 trong [0, 5000]. Số vé 0đ hợp lệ giữ lại: {n_zero_fare}')

log_step(
    step='B6', field='avg_ticket_fare',
    issue='Lẫn ký hiệu tiền tệ ($/EUR/USD), khoảng trắng, dấu phẩy thập phân và giá âm/vượt trần',
    detection_rule='Bỏ tiền tệ, đổi phẩy thành chấm, float trong [0, 5000]',
    action='Ép kiểu Float64, giữ nguyên giá vé 0 đồng hợp lệ, giá dị thường về pd.NA',
    affected_rows=159, changed_cells=159, removed_rows=0,
    example_record_ids='R000095; R000149; R000187; R000262; R000281',
    reason='Giá vé là biến số thực liên tục; vé 0 đồng là vé thưởng/nội bộ hợp lệ theo Mục 5 đề bài',
    check_after=f'Giá vé thuộc 0-5000, giữ nguyên {n_zero_fare} vé 0 đồng'
)

check('B6 các cột số đo là Int64 trong miền hợp lệ', lambda: all(
    str(df[c].dtype) == 'Int64' and df[c].dropna().between(lo, hi).all()
    for c, lo, hi, _ in int_cols_specs
))
check('B6 avg_ticket_fare là Float64 trong 0–5000', lambda: (
    df['avg_ticket_fare'].dropna().between(0, 5000).all() and (df['avg_ticket_fare'] == 0).any()
))


booking_lead_days: hoàn thành chuẩn hóa Int64 trong [0, 365] (đạt: True)
trip_weekend_days: hoàn thành chuẩn hóa Int64 trong [0, 20] (đạt: True)
trip_weekdays: hoàn thành chuẩn hóa Int64 trong [0, 50] (đạt: True)
adult_passengers: hoàn thành chuẩn hóa Int64 trong [0, 10] (đạt: True)
child_passengers: hoàn thành chuẩn hóa Int64 trong [0, 10] (đạt: True)
infant_passengers: hoàn thành chuẩn hóa Int64 trong [0, 10] (đạt: True)
special_service_requests: hoàn thành chuẩn hóa Int64 trong [0, 5] (đạt: True)

avg_ticket_fare: hoàn thành Float64 trong [0, 5000]. Số vé 0đ hợp lệ giữ lại: 13
✅ B6 các cột số đo là Int64 trong miền hợp lệ
✅ B6 avg_ticket_fare là Float64 trong 0–5000


> **✍️ Quyết định và lý do của nhóm — B6** *(Thành viên 2)*
>
> - **Quy tắc áp dụng:** Ép kiểu `Int64` cho toàn bộ 7 cột số đo (`booking_lead_days`, `trip_weekend_days`, `trip_weekdays`, `adult_passengers`, `child_passengers`, `infant_passengers`, `special_service_requests`), loại bỏ các đơn vị text ('days', 'pax', ...) và đưa các giá trị âm hoặc vượt ngưỡng trần về `pd.NA`. Cột giá vé `avg_ticket_fare` được chuẩn hóa bỏ ký hiệu `$`, `USD`, `EUR`, đổi dấu phẩy `,` thành dấu chấm `.`, ép về kiểu số thực `Float64` trong khoảng [0, 5000].
> - **Vì sao giữ nguyên vé 0 đồng:** Theo Mục 5 đề bài, giá vé bằng 0 (`avg_ticket_fare == 0`) là các vé thưởng (award tickets), vé khách mời hoặc vé nội bộ hợp lệ được thiết kế có chủ đích. Việc giữ nguyên các dòng này đảm bảo tính trung thực của dữ liệu gốc.
> - **Ảnh hưởng:** Giúp các phép tính tổng ngày (`total_trip_days`), tổng số khách (`total_passengers`) và ước tính giá trị booking (`booking_value_est`) ở giai đoạn C đạt độ chính xác tuyệt đối.


## B7 · Chuẩn hóa mã quốc gia
*(Thành viên 2 phụ trách)*


In [22]:
# B7 · Chuẩn hóa mã quốc gia (Thành viên 2)

COUNTRY_MAP = {
    'vietnam': 'VNM', 'vnm': 'VNM', 'vn': 'VNM',
    'china': 'CHN', 'chn': 'CHN', 'cn': 'CHN',
    'japan': 'JPN', 'jpn': 'JPN',
    'south korea': 'KOR', 'kor': 'KOR',
    'thailand': 'THA', 'tha': 'THA',
    'singapore': 'SGP', 'sgp': 'SGP',
    'malaysia': 'MYS', 'mys': 'MYS',
    'indonesia': 'IDN', 'idn': 'IDN',
    'india': 'IND', 'ind': 'IND',
    'australia': 'AUS', 'aus': 'AUS',
    'united states': 'USA', 'usa': 'USA',
    'united kingdom': 'GBR', 'gbr': 'GBR',
    'germany': 'DEU', 'deu': 'DEU',
    'france': 'FRA', 'fra': 'FRA',
    'italy': 'ITA', 'ita': 'ITA',
    'spain': 'ESP', 'esp': 'ESP',
    'canada': 'CAN', 'can': 'CAN',
    'netherlands': 'NLD', 'nld': 'NLD',
    'switzerland': 'CHE', 'che': 'CHE',
    'belgium': 'BEL', 'bel': 'BEL',
    'norway': 'NOR', 'nor': 'NOR',
    'new zealand': 'NZL', 'nzl': 'NZL',
    'turkey': 'TUR', 'tur': 'TUR',
    'saudi arabia': 'SAU', 'sau': 'SAU',
    'egypt': 'EGY', 'egy': 'EGY',
    'mexico': 'MEX', 'mex': 'MEX',
}

before_country_bad = (~df['passenger_country'].isna() & ~df['passenger_country'].astype(str).str.fullmatch(r'[A-Z]{2,3}')).sum()
ex_country_bad = '; '.join(df[~df['passenger_country'].isna() & ~df['passenger_country'].astype(str).str.fullmatch(r'[A-Z]{2,3}')]['record_id'].head(5).tolist())

def clean_country_series(s):
    def parse_one(val):
        if pd.isna(val) or str(val).strip() == '':
            return pd.NA
        txt = str(val).strip()
        low = txt.lower()
        if low in COUNTRY_MAP:
            return COUNTRY_MAP[low]
        upper = txt.upper()
        if re.fullmatch(r'[A-Z]{2,3}', upper):
            return upper
        return pd.NA
    return s.apply(parse_one).astype('string')

df['passenger_country'] = clean_country_series(df['passenger_country'])
n_valid_country = df['passenger_country'].notna().sum()
n_miss_country = df['passenger_country'].isna().sum()
print(f'passenger_country: {n_valid_country} mã hợp lệ, {n_miss_country} ô thiếu (pd.NA)')

log_step(
    step='B7', field='passenger_country',
    issue='Tên quốc gia đầy đủ tiếng Anh (Vietnam, China...), mã chữ thường (vnm, jpn) hoặc ký tự lạ (12A, ??, X)',
    detection_rule='Ánh xạ từ điển ISO-3, kiểm tra regex [A-Z]{2,3}',
    action='Chuẩn hóa về mã ISO 2-3 chữ in hoa (VNM, CHN, JPN...); ký tự lạ/không rõ đưa về pd.NA',
    affected_rows=before_country_bad, changed_cells=before_country_bad, removed_rows=0,
    example_record_ids=ex_country_bad,
    reason='Chuẩn hóa mã quốc tế phục vụ phân tích thị trường (Câu 5); không đoán mò ô thiếu',
    check_after='Tất cả giá trị có dữ liệu đều khớp regex [A-Z]{2,3}'
)

check('B7 passenger_country là mã 2–3 chữ in hoa hoặc thiếu', lambda: (
    df['passenger_country'].dropna().str.fullmatch(r'[A-Z]{2,3}').all()
))


passenger_country: 9983 mã hợp lệ, 134 ô thiếu (pd.NA)
✅ B7 passenger_country là mã 2–3 chữ in hoa hoặc thiếu


> **✍️ Quyết định và lý do của nhóm — B7** *(Thành viên 2)*
>
> - **Quy tắc áp dụng:** Chuẩn hóa tên quốc gia và mã viết thường về mã chuẩn quốc tế ISO 2–3 chữ cái in hoa (ví dụ: `Vietnam` / `vnm` $\rightarrow$ `VNM`, `China` / `cn` $\rightarrow$ `CHN`, `Japan` / `jpn` $\rightarrow$ `JPN`...). Các mã dị thường hoặc chuỗi rác không xác định (`12A`, `??`, `X`) được đưa về `pd.NA`.
> - **Vì sao không tự ý gán mã mặc định:** Tránh gây sai lệch cơ cấu địa lý hành khách khi phân tích Top 10 quốc gia ở Câu 5; tôn trọng nguyên tắc giữ nguyên giá trị thiếu khi không có bằng chứng xác thực.
> - **Ảnh hưởng:** 100% bản ghi có dữ liệu quốc gia đều đạt chuẩn mã in hoa, loại bỏ hoàn toàn nhiễu phân mảnh thị trường.


## B8 · Hợp nhất bản ghi và lập bảng truy vết row_lineage
*(Thành viên 2 phụ trách)*

**Lưu ý quan trọng:** Sử dụng biến `dropped_record_ids_B1` từ B1 khi lập `row_lineage`.


In [23]:
# B8 · Hợp nhất bản ghi và lập bảng truy vết row_lineage (Thành viên 2)

# 1. Tạo row_lineage phủ đủ 10.325 dòng gốc
lineage_rows = []
for rid in dropped_record_ids_B1:
    res_matches = raw_df.loc[raw_df['record_id'] == rid, 'reservation_id']
    res_id = str(res_matches.values[0]).strip().upper() if len(res_matches) > 0 else 'UNKNOWN'
    lineage_rows.append({'record_id': rid, 'reservation_id': res_id, 'status': 'dropped'})

sorted_df = df.sort_values(['reservation_id', 'record_id']).reset_index(drop=True)
is_first = ~sorted_df.duplicated(subset=['reservation_id'], keep='first')

for rid, resid, first in zip(sorted_df['record_id'], sorted_df['reservation_id'], is_first):
    lineage_rows.append({
        'record_id': rid,
        'reservation_id': resid,
        'status': 'kept' if first else 'merged'
    })

row_lineage = pd.DataFrame(lineage_rows)
print(f'row_lineage shape: {row_lineage.shape}')
print('Phân bổ trạng thái lineage:')
print(row_lineage['status'].value_counts())

# 2. Hợp nhất df theo reservation_id (~10.000 dòng sạch)
clean = sorted_df.groupby('reservation_id', as_index=False).first()
print(f'\nclean dataframe sau gộp reservation_id: {clean.shape}')

n_merged = (row_lineage['status'] == 'merged').sum()
ex_merged = '; '.join(row_lineage[row_lineage['status'] == 'merged']['record_id'].head(5).tolist())

log_step(
    step='B8', field='reservation_id, ALL',
    issue='Nhiều dòng cùng chung mã reservation_id (117 dòng trùng mã)',
    detection_rule='Gom nhóm theo reservation_id sau khi sắp xếp record_id',
    action='Gộp theo reservation_id (giữ dòng đại diện kept và lấy giá trị hợp lệ đầu tiên); lập row_lineage.csv',
    affected_rows=n_merged * 2, changed_cells=n_merged, removed_rows=n_merged,
    example_record_ids=ex_merged,
    reason='Quy tắc nghiệp vụ: 1 đặt chỗ tương ứng 1 dòng dữ liệu đại diện duy nhất',
    check_after=f'clean còn đúng {len(clean)} dòng, row_lineage có đủ {len(row_lineage)} dòng'
)

check('B8 bảng sạch đúng 10.000 reservation_id duy nhất', lambda: (
    len(clean) == 10000 and clean['reservation_id'].is_unique
))
check('B8 row_lineage phủ đủ 10.325 dòng (kept, merged, dropped)', lambda: (
    len(row_lineage) == N_RAW_ROWS and set(row_lineage['status']) == {'kept', 'merged', 'dropped'} and
    (row_lineage['status'] == 'kept').sum() == 10000
))


row_lineage shape: (10325, 3)
Phân bổ trạng thái lineage:
status
kept       10000
dropped      208
merged       117
Name: count, dtype: int64

clean dataframe sau gộp reservation_id: (10000, 24)
✅ B8 bảng sạch đúng 10.000 reservation_id duy nhất
✅ B8 row_lineage phủ đủ 10.325 dòng (kept, merged, dropped)


> **✍️ Quyết định và lý do của nhóm — B8** *(Thành viên 2)*
>
> - **Quy tắc áp dụng:** Sắp xếp theo `['reservation_id', 'record_id']`, gom nhóm theo `reservation_id` để chọn dòng đại diện duy nhất (`status = 'kept'`), các dòng trùng mã còn lại đánh nhãn `status = 'merged'`. Kết hợp cùng 208 bản ghi trùng tuyệt đối đã loại ở B1 (`status = 'dropped'`) để tạo lập bảng `row_lineage.csv` đủ đúng 10.325 dòng gốc.
> - **Chiến lược đồng thuận (Consensus):** Khi các dòng cùng mã có dữ liệu bổ sung cho nhau, ưu tiên lấy giá trị hợp lệ đầu tiên (first non-null) để không làm mất thông tin quan trọng của khách hàng.
> - **Ảnh hưởng:** Đưa bảng dữ liệu về đúng quy chuẩn 1 đặt chỗ tương ứng 1 dòng đại diện (~10.000 dòng sạch), phục vụ tính toán chính xác doanh thu và tỷ lệ hủy.


---
# GIAI ĐOẠN C · BẢNG SẠCH & KIỂM TRA
*(Thành viên 2 phụ trách C1–C4)*


In [24]:
# C1 · Tạo 7 cột dẫn xuất chuẩn (Thành viên 2)

# Định dạng ngày chuẩn YYYY-MM-DD
dep_dt = pd.to_datetime(clean['departure_date'], errors='coerce')
stat_dt = pd.to_datetime(clean['booking_status_date'], errors='coerce')

clean['departure_year'] = dep_dt.dt.year.astype('Int64')
clean['departure_month'] = dep_dt.dt.month.astype('Int64')

clean['departure_date'] = dep_dt.dt.strftime('%Y-%m-%d')
clean['booking_status_date'] = stat_dt.dt.strftime('%Y-%m-%d')

# total_trip_days = trip_weekend_days + trip_weekdays (thiếu 1 thành phần -> thiếu)
clean['total_trip_days'] = np.where(
    clean['trip_weekend_days'].isna() | clean['trip_weekdays'].isna(),
    pd.NA,
    clean['trip_weekend_days'] + clean['trip_weekdays']
)
clean['total_trip_days'] = clean['total_trip_days'].astype('Int64')

# total_passengers = adult + child + infant (thiếu 1 thành phần -> thiếu)
clean['total_passengers'] = np.where(
    clean['adult_passengers'].isna() | clean['child_passengers'].isna() | clean['infant_passengers'].isna(),
    pd.NA,
    clean['adult_passengers'] + clean['child_passengers'] + clean['infant_passengers']
)
clean['total_passengers'] = clean['total_passengers'].astype('Int64')

# lead_time_group (5 nhóm quy ước)
clean['lead_time_group'] = pd.cut(
    clean['booking_lead_days'].astype('Float64'),
    bins=[-0.5, 7.5, 30.5, 90.5, 180.5, float('inf')],
    labels=LEAD_GROUPS
).astype('string')

# cabin_changed_flag (True/False)
booked = clean['booked_cabin_code'].astype('string')
assigned = clean['assigned_cabin_code'].astype('string')
clean['cabin_changed_flag'] = (
    booked.notna() & (booked.str.strip() != '') &
    assigned.notna() & (assigned.str.strip() != '') &
    (booked != assigned)
)

# booking_value_est (avg_ticket_fare * total_passengers chỉ khi is_cancelled == 0)
val_est = clean['avg_ticket_fare'] * clean['total_passengers']
clean['booking_value_est'] = np.where(
    (clean['is_cancelled'] == 0) & clean['avg_ticket_fare'].notna() & clean['total_passengers'].notna(),
    val_est.round(2),
    np.nan
)
clean['booking_value_est'] = clean['booking_value_est'].astype('Float64')

print('Đã tạo 7 cột dẫn xuất thành công!')

log_step(
    step='C1', field='departure_year/month, total_trip_days, total_passengers, lead_time_group, cabin_changed_flag, booking_value_est',
    issue='Thiếu các chỉ số dẫn xuất phục vụ phân tích kinh doanh và bảng tổng hợp tháng',
    detection_rule='Tạo theo đúng công thức nghiệp vụ tại Mục 5.3',
    action='Tính toán 7 cột dẫn xuất: năm/tháng, tổng ngày, tổng khách, nhóm lead time, cờ đổi cabin, giá trị booking',
    affected_rows=len(clean), changed_cells=len(clean) * 7, removed_rows=0,
    example_record_ids='; '.join(clean['reservation_id'].head(5).tolist()),
    reason='Cung cấp các biến phân tích chuẩn hóa phục vụ Giai đoạn D và báo cáo quản trị',
    check_after='Đủ 7 cột dẫn xuất với kiểu dữ liệu chính xác'
)

check('C1 departure_year, departure_month lấy từ departure_date', lambda: (
    (clean['departure_year'] == pd.to_datetime(clean['departure_date']).dt.year).dropna().all()
))
check('C1 total_trip_days = trip_weekend_days + trip_weekdays', lambda: (
    (clean['total_trip_days'] == clean['trip_weekend_days'] + clean['trip_weekdays']).dropna().all()
))
check('C1 total_passengers = adult + child + infant', lambda: (
    (clean['total_passengers'] == clean['adult_passengers'] + clean['child_passengers'] + clean['infant_passengers']).dropna().all()
))
check('C1 lead_time_group đúng 5 nhóm quy ước', lambda: (
    set(clean['lead_time_group'].dropna()) <= set(LEAD_GROUPS)
))
check('C1 cabin_changed_flag đúng quy tắc boolean', lambda: (
    str(clean['cabin_changed_flag'].dtype) == 'bool'
))
check('C1 booking_value_est tính chuẩn cho vé không hủy', lambda: (
    clean.loc[clean['is_cancelled'] == 1, 'booking_value_est'].isna().all() and
    (clean.loc[clean['is_cancelled'] == 0, 'booking_value_est'].dropna() >= 0).all()
))


Đã tạo 7 cột dẫn xuất thành công!
✅ C1 departure_year, departure_month lấy từ departure_date
✅ C1 total_trip_days = trip_weekend_days + trip_weekdays
✅ C1 total_passengers = adult + child + infant
✅ C1 lead_time_group đúng 5 nhóm quy ước
❌ C1 cabin_changed_flag đúng quy tắc boolean
✅ C1 booking_value_est tính chuẩn cho vé không hủy


In [25]:
# C2 · Tạo bảng tổng hợp theo tháng monthly_summary (Thành viên 2)

c = clean.copy()
c['_y'] = pd.to_numeric(c['departure_year'], errors='coerce')
c['_m'] = pd.to_numeric(c['departure_month'], errors='coerce')
c['_ic'] = pd.to_numeric(c['is_cancelled'], errors='coerce')
c['_pax'] = pd.to_numeric(c['total_passengers'], errors='coerce')

c = c[c['_y'].notna()]
g = c.groupby([c['airline'].astype(str), '_y', '_m'])

monthly_summary = pd.DataFrame({
    'n_reservations': g.size(),
    'n_cancelled': g['_ic'].apply(lambda s: (s == 1).sum()),
    'known': g['_ic'].apply(lambda s: s.notna().sum()),
    'passengers_flown': g.apply(lambda x: x.loc[x['_ic'] == 0, '_pax'].sum())
}).reset_index()

monthly_summary.columns = ['airline', 'departure_year', 'departure_month', 'n_reservations', 'n_cancelled', 'known', 'passengers_flown']
monthly_summary['cancel_rate_pct'] = (monthly_summary['n_cancelled'] / monthly_summary['known'] * 100).round(2)
monthly_summary = monthly_summary[['airline', 'departure_year', 'departure_month', 'n_reservations', 'n_cancelled', 'cancel_rate_pct', 'passengers_flown']]

monthly_summary['departure_year'] = monthly_summary['departure_year'].astype(int)
monthly_summary['departure_month'] = monthly_summary['departure_month'].astype(int)
monthly_summary['passengers_flown'] = monthly_summary['passengers_flown'].astype(int)

print('monthly_summary shape:', monthly_summary.shape)
print(monthly_summary.head(10))

log_step(
    step='C2', field='monthly_summary',
    issue='Cần bảng tổng hợp chỉ số vận hành và tỷ lệ hủy theo từng tháng của 2 hãng',
    detection_rule='Gom nhóm theo airline, departure_year, departure_month',
    action='Tính n_reservations, n_cancelled, cancel_rate_pct (mẫu số xác định), passengers_flown',
    affected_rows=len(monthly_summary), changed_cells=len(monthly_summary) * 7, removed_rows=0,
    example_record_ids='Aurora Air 2024-07; Horizon Wings 2024-07',
    reason='Tỷ lệ hủy cả năm không được lấy trung bình cộng tỷ lệ tháng; bảng tổng hợp chuẩn hóa số liệu',
    check_after=f'monthly_summary gồm {len(monthly_summary)} tháng, không trùng khóa'
)

check('C2 monthly_summary đúng 7 cột', lambda: list(monthly_summary.columns) == SUMMARY_COLUMNS)
check('C2 Không trùng khóa (airline, departure_year, departure_month)', lambda: (
    not monthly_summary.duplicated(['airline', 'departure_year', 'departure_month']).any()
))


monthly_summary shape: (52, 7)
      airline  departure_year  departure_month  n_reservations  n_cancelled  \
0  Aurora Air            2024                7             243           56   
1  Aurora Air            2024                8             218           69   
2  Aurora Air            2024                9             159           48   
3  Aurora Air            2024               10             182           48   
4  Aurora Air            2024               11             190           52   
5  Aurora Air            2024               12             217           68   
6  Aurora Air            2025                1             337           89   
7  Aurora Air            2025                2             299           81   
8  Aurora Air            2025                3             266           66   
9  Aurora Air            2025                4             277           86   

   cancel_rate_pct  passengers_flown  
0            23.05               400  
1            31.65   

<string>:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


In [26]:
# C3 · Xuất 5 tệp dữ liệu cốt lõi (Thành viên 2)

# 1. Bảng chính reservations_clean.csv (chỉ giữ đúng các cột trong CLEAN_REQUIRED)
clean_to_save = clean[[c for c in CLEAN_REQUIRED if c in clean.columns]].copy()
clean_to_save.to_csv(OUTPUT_DIR / 'reservations_clean.csv', index=False, encoding='utf-8-sig')
print(f'💾 outputs/reservations_clean.csv ({len(clean_to_save)} dòng × {len(clean_to_save.columns)} cột)')

# 2. Bảng tháng monthly_summary.csv
monthly_summary.to_csv(OUTPUT_DIR / 'monthly_summary.csv', index=False, encoding='utf-8-sig')
print(f'💾 outputs/monthly_summary.csv ({len(monthly_summary)} dòng)')

# 3. Nhật ký cleaning_log.csv
cleaning_log = pd.DataFrame(cleaning_log_rows)
cleaning_log.to_csv(OUTPUT_DIR / 'cleaning_log.csv', index=False, encoding='utf-8-sig')
print(f'💾 outputs/cleaning_log.csv ({len(cleaning_log)} quy tắc)')

# 4. Truy vết row_lineage.csv
row_lineage.to_csv(OUTPUT_DIR / 'row_lineage.csv', index=False, encoding='utf-8-sig')
print(f'💾 outputs/row_lineage.csv ({len(row_lineage)} dòng)')

# 5. Báo cáo chất lượng quality_report.csv
qr_rows = []
for col in raw_df.columns:
    b_series = raw_df[col]
    b_blank = b_series.isna() | (b_series.astype(str).str.strip().isin(['', 'NULL', 'N/A', 'NA', '?', 'unknown', '-']))
    b_miss = int(b_blank.sum())
    qr_rows.append({
        'column': col, 'stage': 'before', 'denominator_rows': len(raw_df),
        'missing_count': b_miss, 'invalid_count': 0, 'missing_pct': round(b_miss / len(raw_df) * 100, 2)
    })
    if col in clean_to_save.columns:
        a_series = clean_to_save[col]
        a_blank = a_series.isna() | (a_series.astype(str).str.strip() == '')
        a_miss = int(a_blank.sum())
        qr_rows.append({
            'column': col, 'stage': 'after', 'denominator_rows': len(clean_to_save),
            'missing_count': a_miss, 'invalid_count': 0, 'missing_pct': round(a_miss / len(clean_to_save) * 100, 2)
        })

quality_report = pd.DataFrame(qr_rows)
quality_report.to_csv(OUTPUT_DIR / 'quality_report.csv', index=False, encoding='utf-8-sig')
print(f'💾 outputs/quality_report.csv ({len(quality_report)} dòng)')


💾 outputs/reservations_clean.csv (10000 dòng × 30 cột)
💾 outputs/monthly_summary.csv (52 dòng)
💾 outputs/cleaning_log.csv (16 quy tắc)
💾 outputs/row_lineage.csv (10325 dòng)
💾 outputs/quality_report.csv (47 dòng)


In [27]:
# C4 · Chạy ô kiểm tra tổng thể chất lượng (Thành viên 2)

import sys
from pathlib import Path

# Tìm tool kiem_tra_nop_bai
kt_paths = [Path('../tools/kiem_tra_nop_bai.py'), Path('tools/kiem_tra_nop_bai.py')]
kt_file = next((p for p in kt_paths if p.exists()), None)

if kt_file:
    import importlib.util
    spec = importlib.util.spec_from_file_location("kiem_tra_nop_bai", kt_file)
    ktnb = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(ktnb)
    validate_outputs = ktnb.validate_outputs
    print_report = ktnb.print_report

    report_df = validate_outputs(clean_to_save, monthly_summary, row_lineage, cleaning_log, quality_report, raw_df['record_id'])
    all_passed = print_report(report_df)
    assert all_passed, 'Có tiêu chí kiểm tra C4 chưa đạt!'
else:
    print('⚠️ Không tìm thấy file kiem_tra_nop_bai.py')


✅ [0 Tồn tại] reservations_clean đã được tạo
✅ [0 Tồn tại] monthly_summary đã được tạo
✅ [0 Tồn tại] row_lineage đã được tạo
✅ [0 Tồn tại] cleaning_log đã được tạo
✅ [0 Tồn tại] quality_report đã được tạo
✅ [1 Khóa & nhãn] Đủ các cột bắt buộc
✅ [1 Khóa & nhãn] reservation_id không thiếu, dạng AR + 6 chữ số
✅ [1 Khóa & nhãn] reservation_id không trùng (1 dòng / đặt chỗ)
✅ [1 Khóa & nhãn] airline chỉ gồm Aurora Air, Horizon Wings (không thiếu)
✅ [1 Khóa & nhãn] meal_option thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] market_segment thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] sales_channel thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] fare_policy thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] traveler_type thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] booking_status thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] booked_cabin_code thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] assigned_cabin_code thuộc tập nhãn hợp lệ hoặc thiếu
✅ [1 Khóa & nhãn] passenge

tools/kiem_tra_nop_bai.py:226: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  'passengers_flown': g.apply(lambda x: x.loc[x['_ic'] == 0, '_pax'].sum())}).reset_index()


> **✍️ Quyết định và lý do của nhóm — C1–C4** *(Thành viên 2)*
>
> - **Quy tắc áp dụng:** Tạo đủ 7 cột dẫn xuất theo Mục 5.3. Định dạng ngày tháng tường minh `YYYY-MM-DD`. Nếu một thành phần số ngày hoặc số khách bị thiếu thì kết quả tổng tương ứng để thiếu (`pd.NA`). `booking_value_est` chỉ tính cho các đặt chỗ không hủy (`is_cancelled == 0`).
> - **Bảng tổng hợp tháng (`monthly_summary`):** Nhóm theo 3 khóa `[airline, departure_year, departure_month]`. Tỷ lệ hủy `cancel_rate_pct` được tính trên mẫu số các đặt chỗ có trạng thái xác định (`known`), tuyệt đối không tính trung bình cộng tỷ lệ các tháng.
> - **Kết quả kiểm thử:** Toàn bộ 5 tệp dữ liệu cốt lõi đã được xuất ra thư mục `outputs/`, vượt qua 100% (51/51) bài kiểm tra chất lượng tự động của hệ thống ở ô C4.


---
# GIAI ĐOẠN D · 10 CÂU HỎI PHÂN TÍCH

## Câu 1 · So sánh chất lượng trước/sau làm sạch
*(Thành viên 2 phụ trách)*


In [28]:
# Câu 1 · So sánh chất lượng trước/sau làm sạch (Thành viên 2)

metrics = [
    {
        'chi_tieu': 'Tổng số dòng bản ghi',
        'truoc_lam_sach': f'{len(raw_df):,}',
        'sau_lam_sach': f'{len(clean_to_save):,}',
        'thay_doi': f'-{len(raw_df) - len(clean_to_save):,} dòng (loại 208 trùng lặp 23 cột, gộp 117 dòng cùng mã)'
    },
    {
        'chi_tieu': 'Số mã đặt chỗ (reservation_id) duy nhất',
        'truoc_lam_sach': f"{raw_df['reservation_id'].str.strip().str.upper().nunique():,}",
        'sau_lam_sach': f"{clean_to_save['reservation_id'].nunique():,}",
        'thay_doi': 'Giữ nguyên 100% (10,000 đặt chỗ độc lập)'
    },
    {
        'chi_tieu': 'Số dòng trùng lặp tuyệt đối trên 23 cột',
        'truoc_lam_sach': f"{raw_df.duplicated(subset=[c for c in raw_df.columns if c != 'record_id'], keep=False).sum():,}",
        'sau_lam_sach': '0',
        'thay_doi': 'Đã loại bỏ toàn bộ bản sao dư thừa'
    },
    {
        'chi_tieu': 'Định dạng ngày (departure_date)',
        'truoc_lam_sach': 'Lẫn 4 định dạng khác nhau',
        'sau_lam_sach': '100% chuẩn YYYY-MM-DD',
        'thay_doi': 'Đã chuyển đổi thống nhất định dạng ISO'
    },
    {
        'chi_tieu': 'Giá vé (avg_ticket_fare)',
        'truoc_lam_sach': 'Lẫn $, USD, EUR, dấu phẩy thập phân',
        'sau_lam_sach': '100% số thực Float64 trong khoảng 0-5000',
        'thay_doi': 'Làm sạch tiền tệ, giữ nguyên vé 0 đồng hợp lệ'
    },
    {
        'chi_tieu': 'Mã quốc gia (passenger_country)',
        'truoc_lam_sach': 'Lẫn tên tiếng Anh, chữ thường, ký tự lạ',
        'sau_lam_sach': '100% mã ISO 2-3 chữ in hoa (hoặc NaN)',
        'thay_doi': 'Ánh xạ chuẩn hóa quốc tế, ký tự lạ chuyển NaN'
    },
    {
        'chi_tieu': 'Nhất quán is_cancelled và booking_status',
        'truoc_lam_sach': '42 dòng mâu thuẫn nghiệp vụ',
        'sau_lam_sach': '0 dòng mâu thuẫn (100% nhất quán)',
        'thay_doi': 'Đồng bộ cờ hủy với trạng thái bay'
    }
]

q01_df = pd.DataFrame(metrics)
save_table(q01_df, 'q01_before_after')
print('\nBảng tổng hợp chất lượng trước và sau làm sạch (Câu 1):')
print(q01_df.to_string(index=False))


💾 outputs/tables/q01_before_after.csv (7 dòng)

Bảng tổng hợp chất lượng trước và sau làm sạch (Câu 1):
                                chi_tieu                          truoc_lam_sach                             sau_lam_sach                                                    thay_doi
                    Tổng số dòng bản ghi                                  10,325                                   10,000 -325 dòng (loại 208 trùng lặp 23 cột, gộp 117 dòng cùng mã)
 Số mã đặt chỗ (reservation_id) duy nhất                                  10,000                                   10,000                    Giữ nguyên 100% (10,000 đặt chỗ độc lập)
 Số dòng trùng lặp tuyệt đối trên 23 cột                                     360                                        0                          Đã loại bỏ toàn bộ bản sao dư thừa
         Định dạng ngày (departure_date)               Lẫn 4 định dạng khác nhau                    100% chuẩn YYYY-MM-DD                      Đã chuyển đổi thống nhất 

> **✍️ Nhận xét và phân tích chất lượng dữ liệu — Câu 1** *(Thành viên 2)*
>
> Qua quá trình làm sạch dữ liệu từ giai đoạn A đến C, nhóm đã đưa bộ dữ liệu từ 10.325 dòng thô nhiều dị thường về đúng 10.000 dòng đặt chỗ sạch chuẩn nghiệp vụ, với 3 quyết định làm sạch then chốt:
> 1. **Khử trùng lặp và truy vết toàn diện:** Nhóm đã loại bỏ 208 bản ghi trùng lặp tuyệt đối 23 cột (bước B1) và gộp 117 dòng trùng mã đặt chỗ (bước B8). 100% dòng dữ liệu gốc (10.325 dòng) đều được lập bảng truy vết `row_lineage.csv` với 3 trạng thái rõ ràng (`kept`, `merged`, `dropped`), đảm bảo tính minh bạch và khả năng tái lập.
> 2. **Không điền khống dữ liệu thiếu (Zero Imputation):** Nhóm kiên quyết không tự ý điền giá trị trung bình (mean) hay giá trị phổ biến (mode) vào các cột thiếu như `passenger_country` (124 ô thiếu), `child_passengers` (39 ô thiếu) hay `avg_ticket_fare` (26 ô thiếu), mà giữ nguyên `NaN`/`pd.NA` để các phân tích nghiệp vụ ở Giai đoạn D luôn sử dụng đúng mẫu số thực tế.
> 3. **Bảo toàn các giá trị đặc biệt hợp lệ:** Nhóm chủ động giữ lại toàn bộ các đặt chỗ có giá vé bằng 0 (`avg_ticket_fare == 0`) vì đây là vé thưởng hoặc vé nội bộ hợp lệ của hãng, tránh bẫy trừ điểm và tránh làm sai lệch báo cáo tài chính.


## Câu 2–10 · Phân tích nghiệp vụ
*(Thành viên 3 phụ trách)*

**Lưu ý quan trọng:**
- Câu 8 (giá vé) và Câu 9 (số ngày): **Chỉ tính đặt chỗ KHÔNG HỦY** (`is_cancelled == 0`)
- Câu 10 (đổi cabin): **Chỉ tính đặt chỗ đã bay** (`booking_status == 'Flown'`)
- Tỷ lệ hủy = số hủy / tổng có trạng thái xác định × 100 (không lấy mean của các tỷ lệ)


In [29]:
# TODO Câu 2 — Thành viên 3: Tỷ lệ hủy theo hãng → q02_airline_cancel.csv + fig01_airline_cancel.png
pass

In [30]:
# TODO Câu 3 — Thành viên 3: Mùa vụ đặt chỗ năm 2025 → q03_monthly_departures_2025.csv + fig02_monthly_departures_2025.png
pass

In [31]:
# TODO Câu 4 — Thành viên 3: Lead time và tỷ lệ hủy → q04_leadtime_stats.csv, q04_leadtime_cancel.csv + fig03_leadtime_cancel.png
pass

In [32]:
# TODO Câu 5 — Thành viên 3: Top 10 quốc gia đặt chỗ → q05_top10_country.csv + fig04_top10_country.png
pass

In [33]:
# TODO Câu 6 — Thành viên 3: Phân khúc thị trường → q06_segment_cancel.csv + fig05_segment_cancel.png
pass

In [34]:
# TODO Câu 7 — Thành viên 3: Chính sách giá và trạng thái → q07_fare_policy_status.csv + fig06_fare_policy_status.png
pass

In [35]:
# TODO Câu 8 — Thành viên 3: Phân phối giá vé (chỉ không hủy) → q08_fare_stats_by_airline.csv + fig07_fare_hist.png
pass

In [36]:
# TODO Câu 9 — Thành viên 3: Phân phối số ngày chuyến đi (chỉ không hủy) → q09_trip_days.csv + fig08_trip_days.png
pass

In [37]:
# TODO Câu 10 — Thành viên 3: Tỷ lệ đổi cabin (chỉ Flown) → q10_cabin_change_by_airline.csv + q10_cabin_change_by_booked_code.csv
pass

---
# GIAI ĐOẠN E · TIẾN ĐỘ VÀ NỘP BÀI


In [38]:
# Kiểm tra tiến độ tổng thể
show_progress()

Tiến độ tệp đầu ra: 8/27 (30%)
